## Preparación de datos

Para reproducir este cuaderno, **debe** existir una carpeta llamada `data` en la raíz del directorio donde se esté ejecutando. En su interior, deben ubicarse **únicamente** los archivos ZIP con los conjuntos de datos que se desean analizar siguiendo el formato del Directorio de Unidades Económicas (DENUE) publicado por el INEGI. Las celdas subsiguientes ejecutan código encargado de procesar y extraer los datos y metadatos necesarios. No es necesaria la intervención del ejecutante.

Los datos se pueden descargar en la siguiente dirección web: https://www.inegi.org.mx/app/descarga/?ti=6.
Aquellos utilizados para este estudio comprenden todos los registros disponibles en la plataforma del DENUE hasta septiembre de 2026, que comienzan en 2010. 

In [5]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Los datos del DENUE se descargan masivamente en formato ZIP
import zipfile
from pathlib import Path

# Directorio con los archivos
path = Path("data")
files_in_path = [f for f in path.iterdir() if f.is_file()]

# Descompresión y guardado
for idx, zip_file in enumerate(files_in_path):
    if "denue" not in str(zip_file):
        print(f"Archivo inválido: {str(zip_file)}. Siguiente...")
        continue

    print(f"Procesando {idx}: ", zip_file)
    extract_to = f"{str(zip_file)[:-4]}"

    with zipfile.ZipFile(zip_file, "r") as zipRef:
        zipRef.extractall(extract_to)

In [ ]:
import csv
import io
import re
import shutil
from pathlib import Path

data = Path('data')
extracted = data / 'extracted'
extracted.mkdir(exist_ok=True)

def read_csv(path):
    raw = path.read_bytes()
    for encoding in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            return io.StringIO(raw.decode(encoding))
        except UnicodeDecodeError:
            pass
    raise UnicodeError(path)

def year_for(folder):
    if folder.name == 'denue_03_csv':
        return 2026
    suffix = folder.name.removeprefix('denue_03_').removesuffix('_csv')
    if re.fullmatch(r'20\d{2}', suffix):
        return int(suffix)
    if re.fullmatch(r'\d{8}', suffix):
        return int(suffix[-4:])
    if re.fullmatch(r'\d{4}', suffix):
        return 2000 + int(suffix[-2:])
    raise ValueError(f'Fecha irreconocible en {folder.name}')

sources = []
for folder in sorted(data.glob('denue_*_csv')):
    if not folder.is_dir():
        continue
    files = [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() == '.csv' and 'diccionario_de_datos' not in p.parts]
    if len(files) != 1:
        raise RuntimeError(f'Se esperaba in CSV en {folder}; se encontró {len(files)}')
    sources.append((folder, files[0]))

for folder, source in sources:
    reader = csv.reader(read_csv(source))
    header = next(reader)
    target = extracted / f'{folder.name.removesuffix("_csv")}.csv'
    year = year_for(folder)
    with target.open('w', encoding='utf-8-sig', newline='') as output:
        writer = csv.writer(output, lineterminator='\n')
        writer.writerow([*header, 'YEAR'])
        for line_no, row in enumerate(reader, 2):
            if row:
                if len(row) != len(header):
                    raise RuntimeError(f'{source}:{line_no} tiene un número inválido de campos')
                writer.writerow([*row, year])

dictionaries = [(folder, p) for folder in data.glob('denue_*_csv')
                if folder.is_dir()
                for p in folder.rglob('*') if p.is_file() and p.name.lower() == 'denue_diccionario_de_datos.csv']
dictionary = max(dictionaries, key=lambda item: year_for(item[0]))[1]
rows = list(csv.reader(read_csv(dictionary)))
rows.append(['YEAR', 'YEAR', 'numérico', '4', 'Año de la versión del DENUE usada para el registro.'])
with (extracted / 'diccionario_de_datos.csv').open('w', encoding='utf-8-sig', newline='') as output:
    csv.writer(output, lineterminator='\n').writerows(rows)

files = sorted(extracted.glob('*.csv'))
for source in files:
    source.replace(data / source.name)
shutil.rmtree(extracted)

for path in data.iterdir():
    if path.is_dir():
        shutil.rmtree(path)
    elif path.suffix.lower() != '.csv':
        path.unlink()

print(f'se guardaron {len(files)} archivos `.csv` en {data}/*')

## Lectura

In [7]:
import pandas as pd
from pathlib import Path

# Construir conjunto de datos
csv_files = sorted(Path('data').glob('denue_*.csv'))
base_df = pd.concat((pd.read_csv(csv, encoding='utf-8') for csv in csv_files), ignore_index=True)
base_df = base_df.drop(columns=['clee', 'codigo_act'])

base_df.head()

,id,nom_estab,raz_social,nombre_act,per_ocu,tipo_vial,nom_vial,tipo_v_e_1,nom_v_e_1,tipo_v_e_2,...,Tipo de entre vialidad 2,Nombre de entre vialidad 2,Tipo de entre vialidad 3,Nombre de entre vialidad 3,Letra exterior,Edificio,Edificio Piso,Número interior,Letra interior,Tipo centro comercial
0,12319963.0,NaN,NaN,"Pesca y captura de peces, crustáceos, moluscos...",0 a 5 personas,CALLE,MIGUEL HIDALGO Y COSTILLA,CALLE,ADOLFO LOPEZ MATEOS,BOULEVARD,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12299626.0,NaN,NaN,Otra acuicultura,6 a 10 personas,CALLE,NINGUNO,CALLE,Ninguno,CALLE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12319788.0,NaN,NaN,"Pesca y captura de peces, crustáceos, moluscos...",6 a 10 personas,CALLE,NINGUNO,CALLE,Ninguno,CALLE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12319789.0,NaN,NaN,"Pesca y captura de peces, crustáceos, moluscos...",0 a 5 personas,CALLE,NINGUNO,CALLE,Ninguno,OTRO(ESPECIFIQUE),...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6166554.0,NaN,NaN,"Pesca y captura de peces, crustáceos, moluscos...",11 a 30 personas,CALLE,BALLENA,CALLE,Cardón,CALLE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Análisis de columnas
for column in base_df.columns:
    print(column)

In [12]:
# Verificación de longitud
print("Registros disponibles:", len(base_df))

Registros disponibles: 836935
